In [18]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.base import clone
from sklearn.ensemble import (
    RandomForestClassifier,
    RandomForestRegressor,
    ExtraTreesRegressor,
    HistGradientBoostingRegressor,
    VotingRegressor,
)
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error,
    balanced_accuracy_score,
    confusion_matrix,
)


In [19]:
month_cols = ['Month_1','Month_2','Month_3','Month_4','Month_5','Month_6',
              'Month_7','Month_8','Month_9','Month_10','Month_11','Month_12']
paper_features = ['Latitude_Degrees', 'Longitude_Degrees', 'Depth_m', 'ClimSST', 'SSTA']
extra_cols =  paper_features + month_cols

In [20]:
df = pd.read_csv("best_data.csv")
df = df[extra_cols + ['Percent_Bleaching']]
# สร้าง Status
df['Status'] = np.where(df['Percent_Bleaching'] <= 1, True, False)
df = df.reset_index(drop=True)

In [21]:
RANDOM_STATE = 42
RESERVED_LOW_N = 0


def make_v2_training_sets(train_part, n_reserved=RESERVED_LOW_N, random_state=RANDOM_STATE):
    """Reserve low-bleaching rows from train only for the regressor flow."""
    low_candidates = train_part[train_part['Percent_Bleaching'] <= 1]
    if len(low_candidates) < n_reserved:
        raise ValueError(
            f"Need at least {n_reserved} rows with Percent_Bleaching <= 1 "
            f"in the training partition, found {len(low_candidates)}."
        )

    reserved_low = low_candidates.sample(n=low_candidates.shape[0], random_state=random_state)
    train_cls = train_part.reset_index(drop=True) # train_part.drop(index=reserved_low.index).reset_index(drop=True)
    train_reg = pd.concat([
        train_part[train_part['Percent_Bleaching'] > 1],
        reserved_low,
    ], axis=0).reset_index(drop=True)

    return train_cls, train_reg, reserved_low.reset_index(drop=True)


def fit_pipeline(X_train):
    """Fit imputer + scaler + PCA แล้ว concat กลับด้วย month + paper features (raw)"""
    X = X_train.reset_index(drop=True)
    X_extra   = X[extra_cols].copy()
    X_for_pca = X.drop(columns=month_cols)   # paper features ยังอยู่ใน PCA ตาม flow เดิม

    imputer = SimpleImputer(strategy='median')
    X_imp   = imputer.fit_transform(X_for_pca)

    scaler  = StandardScaler()
    X_ss    = scaler.fit_transform(X_imp)

    pca     = PCA(n_components=1)
    X_pca   = pca.fit_transform(X_ss)

    pc_cols = [f'PC{i+1}' for i in range(X_pca.shape[1])]
    X_final = pd.concat([pd.DataFrame(X_pca, columns=pc_cols), X_extra], axis=1)
    return imputer, scaler, pca, pc_cols, X_final


def transform_pipeline(X, imputer, scaler, pca, pc_cols):
    """ใช้ pipeline ที่ fit แล้ว transform ข้อมูลใหม่"""
    X = X.reset_index(drop=True)
    X_extra   = X[extra_cols].copy()
    X_for_pca = X.drop(columns=month_cols)

    X_imp = imputer.transform(X_for_pca)
    X_ss  = scaler.transform(X_imp)
    X_pca = pca.transform(X_ss)

    X_final = pd.concat([pd.DataFrame(X_pca, columns=pc_cols), X_extra], axis=1)
    return X_final


In [22]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=df['Status'],
)
train_cls, train_reg, reserved_low = make_v2_training_sets(train_df)

# Test set is held out until final evaluation.
X_test = test_df.drop(columns=['Percent_Bleaching', 'Status']).reset_index(drop=True)
y_test = test_df['Percent_Bleaching'].reset_index(drop=True)

# ===== Pipeline A: Classifier (train - reserved low rows) =====
X_train_cls = train_cls.drop(columns=['Percent_Bleaching', 'Status'])
y_train_cls = train_cls['Status'].reset_index(drop=True)

imputer_A, scaler_A, pca_A, pc_cols_A, X_train_cls_final = fit_pipeline(X_train_cls)

rf_cls_param =  {'n_estimators': 587, 'max_depth': 38, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': None}
rf_classifier = RandomForestClassifier(
    **rf_cls_param,
    class_weight='balanced',
    criterion='gini',
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
rf_classifier.fit(X_train_cls_final, y_train_cls)

# ===== Pipeline B: Regressors (>1 rows + reserved low rows) =====
X_train_reg = train_reg.drop(columns=['Percent_Bleaching', 'Status'])
y_train_reg = train_reg['Percent_Bleaching'].reset_index(drop=True)

imputer_B, scaler_B, pca_B, pc_cols_B, X_train_reg_final = fit_pipeline(X_train_reg)


et_param =  {'n_estimators': 298, 'max_depth': 27, 'min_samples_leaf': 1, 'max_features': 'sqrt'}

rf_paran =  {'n_estimators': 219, 'max_depth': 31, 'min_samples_split': 4, 'min_samples_leaf': 1, 'max_features': 'log2'}

hgb_param = {'max_iter': 362, 'max_depth': 11, 'learning_rate': 0.07980078947589887, 'min_samples_leaf': 8, 'l2_regularization': 0.593989343678763}

hgb_model = HistGradientBoostingRegressor(
    **hgb_param,
    random_state=RANDOM_STATE,
)
et_model = ExtraTreesRegressor(
    **et_param,
    n_jobs=-1, 
    random_state=RANDOM_STATE,
)
rf_model = RandomForestRegressor(
    **rf_paran,
    n_jobs=-1, 
    random_state=RANDOM_STATE,
)

hgb_model.fit(X_train_reg_final, y_train_reg)
et_model.fit(X_train_reg_final, y_train_reg)
rf_model.fit(X_train_reg_final, y_train_reg)

voting = VotingRegressor([('hgb', hgb_model), ('et', et_model), ('rf', rf_model)])
voting.fit(X_train_reg_final, y_train_reg)


,"estimators estimators: list of (str, estimator) tuplesInvoking the ``fit`` method on the ``VotingRegressor`` will fit clonesof those original estimators that will be stored in the class attribute``self.estimators_``. An estimator can be set to ``'drop'`` using:meth:`set_params`... versionchanged:: 0.21 ``'drop'`` is accepted. Using None was deprecated in 0.22 and support was removed in 0.24.","[('hgb', ...), ('et', ...), ...]"
,"weights weights: array-like of shape (n_regressors,), default=NoneSequence of weights (`float` or `int`) to weight the occurrences ofpredicted values before averaging. Uses uniform weights if `None`.",None
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for ``fit``.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting will be printed as itis completed... versionadded:: 0.23",False
,"loss loss: {'squared_error', 'absolute_error', 'gamma', 'poisson', 'quantile'}, default='squared_error'The loss function to use in the boosting process. Note that the""squared error"", ""gamma"" and ""poisson"" losses actually implement""half least squares loss"", ""half gamma deviance"" and ""half poissondeviance"" to simplify the computation of the gradient. Furthermore,""gamma"" and ""poisson"" losses internally use a log-link, ""gamma""requires ``y > 0`` and ""poisson"" requires ``y >= 0``.""quantile"" uses the pinball loss... versionchanged:: 0.23 Added option 'poisson'... versionchanged:: 1.1 Added option 'quantile'... versionchanged:: 1.3 Added option 'gamma'.",'squared_error'
,"quantile quantile: float, default=NoneIf loss is ""quantile"", this parameter specifies which quantile to be estimatedand must be between 0 and 1.",None
,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.07980078947589887
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees.",362
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",31
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",11
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",8


In [23]:
perturbation_deltas = {
    'ClimSST':           0.5,     
    'SSTA':              0.05,     
    'Depth_m':           1,      
    'Longitude_Degrees': 0.001,    # ~111 m
    'Latitude_Degrees':  0.001,    
}


def compute_sensitivity_weights_per_row(models, X, deltas, eps=1e-8):
    """Per-row: weight เป็น array shape (n_rows,) ต่อ model"""
    baselines = {name: m.predict(X) for name, m in models.items()}
    n_rows    = len(X)
    errors    = {name: np.zeros(n_rows) for name in models}

    for feat, delta in deltas.items():
        X_plus  = X.copy(); X_plus[feat]  = X_plus[feat]  + delta
        X_minus = X.copy(); X_minus[feat] = X_minus[feat] - delta
        for name, m in models.items():
            base = baselines[name]
            errors[name] += np.abs(m.predict(X_plus)  - base)
            errors[name] += np.abs(m.predict(X_minus) - base)

    # +eps กัน div-by-zero (กรณี row ที่ทุก model ไม่ sensitive เลย)
    inv = {n: 1.0 / (e + eps) for n, e in errors.items()}
    # element-wise normalize ต่อ row
    total = sum(inv.values())                    # shape (n_rows,)
    weights = {n: v / total for n, v in inv.items()}   # แต่ละค่า shape (n_rows,)
    return weights, baselines


def predict_combined(X_test_raw, classifier, base_regs, voting_reg,
                     pipe_A, pipe_B):
    n = len(X_test_raw)

    X_test_A = transform_pipeline(X_test_raw, *pipe_A)
    cls_pred = classifier.predict(X_test_A)

    idx_false = np.where(~cls_pred)[0]

    weights_per_row = None
    baselines       = None

    if len(idx_false) > 0:
        X_subset_raw = X_test_raw.iloc[idx_false].reset_index(drop=True)
        X_subset_B   = transform_pipeline(X_subset_raw, *pipe_B)

        weights_per_row, baselines = compute_sensitivity_weights_per_row(
            base_regs, X_subset_B, perturbation_deltas
        )

    results = {}
    for name, reg in base_regs.items():
        y_pred = np.zeros(n)
        if len(idx_false) > 0:
            y_pred[idx_false] = baselines[name]
        results[name] = y_pred

    # Voting (equal weight)
    y_pred_v = np.zeros(n)
    if len(idx_false) > 0:
        X_subset_B = transform_pipeline(
            X_test_raw.iloc[idx_false].reset_index(drop=True), *pipe_B
        )
        y_pred_v[idx_false] = voting_reg.predict(X_subset_B)
    results['Voting (Ensemble)'] = y_pred_v

    # Sensitivity-Weighted (per-row)
    y_pred_swr = np.zeros(n)
    if len(idx_false) > 0:
        y_pred_swr[idx_false] = sum(
            weights_per_row[n_] * baselines[n_] for n_ in base_regs
        )
    results['Sens-Weighted (per-row)'] = y_pred_swr

    return results, cls_pred, weights_per_row


In [24]:
base_regs = {
    'HistGradientBoost': hgb_model,
    'ExtraTrees':        et_model,
    'RandomForest':      rf_model,
}
pipe_A = (imputer_A, scaler_A, pca_A, pc_cols_A)
pipe_B = (imputer_B, scaler_B, pca_B, pc_cols_B)

results, cls_pred, w_perrow = predict_combined(
    X_test, rf_classifier, base_regs, voting, pipe_A, pipe_B
)

y_test_cls = (y_test <= 1).values
cls_acc = (cls_pred == y_test_cls).mean()
cls_bal_acc = balanced_accuracy_score(y_test_cls, cls_pred)
cls_cm = confusion_matrix(y_test_cls, cls_pred, labels=[True, False])

print("=" * 70)
print("COMBINED FLOW RESULTS (single seed = 42) — V2 reserved-low flow")
print("=" * 70)
print(f"Full cleaned rows: {len(df)}")
print(f"Train rows: {len(train_df)} | Test rows: {len(test_df)}")
print(f"Reserved low rows for regressor only: {len(reserved_low)}")
print(f"Classifier train rows: {len(train_cls)} | Regressor train rows: {len(train_reg)}")
print(f"Regressor train <=1 rows: {(train_reg['Percent_Bleaching'] <= 1).sum()} | >1 rows: {(train_reg['Percent_Bleaching'] > 1).sum()}")
print("-" * 70)
print(f"Classifier accuracy: {cls_acc:.4f}  "
      f"(predicted ≤1: {cls_pred.sum()}/{len(cls_pred)})")
print(f"Classifier balanced accuracy: {cls_bal_acc:.4f}")
print("Confusion matrix rows=true ≤1/>1, cols=pred ≤1/>1")
print(cls_cm)
print("-" * 70)
print(f"{'Regressor':<25}{'R²':>10}{'RMSE':>12}{'MAE':>10}{'Acc%':>10}")
print("-" * 70)

ordered = ['HistGradientBoost', 'ExtraTrees', 'RandomForest',
           'Voting (Ensemble)', 'Sens-Weighted (per-row)']
for name in ordered:
    y_pred = results[name]
    r2   = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)
    print(f"{name:<25}{r2:>10.4f}{rmse:>12.4f}{mae:>10.4f}{100-mae:>10.2f}")
print("=" * 70)

# Per-row stats — ดูการกระจายของ weight ทุก row
if w_perrow:
    print()
    print("Per-row weights (ดู distribution ของแต่ละ model):")
    print(f"{'Model':<22}{'min':>10}{'mean':>10}{'max':>10}{'std':>10}")
    print("-" * 62)
    for name in ['ExtraTrees', 'RandomForest', 'HistGradientBoost']:
        w = w_perrow[name]
        print(f"{name:<22}{w.min():>10.4f}{w.mean():>10.4f}"
              f"{w.max():>10.4f}{w.std():>10.4f}")


COMBINED FLOW RESULTS (single seed = 42) — V2 reserved-low flow
Full cleaned rows: 21928
Train rows: 17542 | Test rows: 4386
Reserved low rows for regressor only: 9670
Classifier train rows: 17542 | Regressor train rows: 17542
Regressor train <=1 rows: 9670 | >1 rows: 7872
----------------------------------------------------------------------
Classifier accuracy: 0.9455  (predicted ≤1: 2415/4386)
Classifier balanced accuracy: 0.9450
Confusion matrix rows=true ≤1/>1, cols=pred ≤1/>1
[[2297  121]
 [ 118 1850]]
----------------------------------------------------------------------
Regressor                        R²        RMSE       MAE      Acc%
----------------------------------------------------------------------
HistGradientBoost            0.6192     13.3021    6.2799     93.72
ExtraTrees                   0.6651     12.4748    5.3010     94.70
RandomForest                 0.6659     12.4598    5.6759     94.32
Voting (Ensemble)            0.6678     12.4252    5.6582     94.34
Sens

In [25]:
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)

method_names = ['HistGradientBoost','ExtraTrees','RandomForest',
                'Voting (Ensemble)','Sens-Weighted (per-row)']
scores = {n: {'r2': [], 'rmse': [], 'mae': []} for n in method_names}
cls_acc_history = []
cls_bal_acc_history = []
fold_count_history = []

# เก็บ stats ของ per-row weights ต่อ fold
perrow_mean_history   = {'ExtraTrees': [], 'RandomForest': [], 'HistGradientBoost': []}
perrow_std_history    = {'ExtraTrees': [], 'RandomForest': [], 'HistGradientBoost': []}

for fold_idx, (train_idx, test_idx) in enumerate(skf.split(df, df['Status']), start=1):
    train_fold = df.iloc[train_idx]
    test_fold  = df.iloc[test_idx]
    train_cls_f, train_reg_f, reserved_low_f = make_v2_training_sets(train_fold)

    X_test_fold     = test_fold.drop(columns=['Percent_Bleaching','Status']).reset_index(drop=True)
    y_test_fold     = test_fold['Percent_Bleaching'].reset_index(drop=True)
    y_test_cls_fold = (y_test_fold <= 1).values

    fold_count_history.append({
        'fold': fold_idx,
        'train': len(train_fold),
        'test': len(test_fold),
        'reserved_low': len(reserved_low_f),
        'classifier_train': len(train_cls_f),
        'regressor_train': len(train_reg_f),
    })

    # Pipeline A
    X_train_cls = train_cls_f.drop(columns=['Percent_Bleaching','Status'])
    y_train_cls = train_cls_f['Status'].reset_index(drop=True)
    imp_A, sc_A, pc_A_, pcA_cols, X_tr_cls = fit_pipeline(X_train_cls)
    cls_fold = clone(rf_classifier)
    cls_fold.fit(X_tr_cls, y_train_cls)

    # Pipeline B
    X_train_reg = train_reg_f.drop(columns=['Percent_Bleaching','Status'])
    y_train_reg = train_reg_f['Percent_Bleaching'].reset_index(drop=True)
    imp_B, sc_B, pc_B_, pcB_cols, X_tr_reg = fit_pipeline(X_train_reg)

    fold_regs = {
        'HistGradientBoost': clone(hgb_model),
        'ExtraTrees':        clone(et_model),
        'RandomForest':      clone(rf_model),
    }
    for m in fold_regs.values():
        m.fit(X_tr_reg, y_train_reg)

    voting_fold = VotingRegressor([
        ('hgb', fold_regs['HistGradientBoost']),
        ('et',  fold_regs['ExtraTrees']),
        ('rf',  fold_regs['RandomForest']),
    ])
    voting_fold.fit(X_tr_reg, y_train_reg)

    pipeA = (imp_A, sc_A, pc_A_, pcA_cols)
    pipeB = (imp_B, sc_B, pc_B_, pcB_cols)
    results, cls_pred, w_perrow = predict_combined(
        X_test_fold, cls_fold, fold_regs, voting_fold, pipeA, pipeB
    )

    cls_acc_history.append((cls_pred == y_test_cls_fold).mean())
    cls_bal_acc_history.append(balanced_accuracy_score(y_test_cls_fold, cls_pred))

    if w_perrow:
        for n_, w_ in w_perrow.items():
            perrow_mean_history[n_].append(w_.mean())
            perrow_std_history[n_].append(w_.std())

    for name, y_pred in results.items():
        scores[name]['r2'].append(r2_score(y_test_fold, y_pred))
        scores[name]['rmse'].append(np.sqrt(mean_squared_error(y_test_fold, y_pred)))
        scores[name]['mae'].append(mean_absolute_error(y_test_fold, y_pred))

# ===== Report =====
print("=" * 75)
print(f"K-Fold Cross Validation Summary (k={n_splits}) — V2 reserved-low flow")
print("=" * 75)
print(f"Mean classifier accuracy: {np.mean(cls_acc_history):.4f}")
print(f"Mean classifier balanced accuracy: {np.mean(cls_bal_acc_history):.4f}")
print("Fold row counts:")
display(pd.DataFrame(fold_count_history))
print("-" * 75)
print(f"{'Regressor':<26}{'Mean R²':>14}{'Mean RMSE':>14}{'Mean MAE':>14}")
print("-" * 75)
for name in method_names:
    s = scores[name]
    print(f"{name:<26}{np.mean(s['r2']):>14.4f}"
          f"{np.mean(s['rmse']):>14.4f}{np.mean(s['mae']):>14.4f}")
print("=" * 75)

print()
print(f"Per-row weights (mean ± std across {n_splits} folds):")
for n_ in ['ExtraTrees','RandomForest','HistGradientBoost']:
    print(f"  {n_:<20}mean={np.mean(perrow_mean_history[n_]):.4f}  "
          f"avg_std={np.mean(perrow_std_history[n_]):.4f}")


K-Fold Cross Validation Summary (k=5) — V2 reserved-low flow
Mean classifier accuracy: 0.9448
Mean classifier balanced accuracy: 0.9437
Fold row counts:


,fold,train,test,reserved_low,classifier_train,regressor_train
0,1,17542,4386,9670,17542,17542
1,2,17542,4386,9670,17542,17542
2,3,17542,4386,9670,17542,17542
3,4,17543,4385,9671,17543,17543
4,5,17543,4385,9671,17543,17543


---------------------------------------------------------------------------
Regressor                        Mean R²     Mean RMSE      Mean MAE
---------------------------------------------------------------------------
HistGradientBoost                 0.6036       13.5583        6.2768
ExtraTrees                        0.6682       12.4037        5.2161
RandomForest                      0.6649       12.4657        5.5776
Voting (Ensemble)                 0.6651       12.4628        5.5873
Sens-Weighted (per-row)           0.6483       12.7689        5.7392

Per-row weights (mean ± std across 5 folds):
  ExtraTrees          mean=0.3125  avg_std=0.1356
  RandomForest        mean=0.3284  avg_std=0.1150
  HistGradientBoost   mean=0.3591  avg_std=0.1710


In [26]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, r2_score, mean_squared_error, mean_absolute_error, balanced_accuracy_score
from sklearn.base import clone

n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)

method_names = ['HistGradientBoost','ExtraTrees','RandomForest',
                'Voting (Ensemble)','Sens-Weighted (per-row)']

# เก็บค่าเฉลี่ยรวมสำหรับสรุปตอนท้าย
final_scores = {n: {
    'r2': [], 'rmse': [], 'mae': [],
    'rmse_pred_0': [], 'rmse_pred_nonzero': [], 'r2_pred_nonzero': []
} for n in method_names}

cls_acc_history = []
conf_matrix_history = []
base_train_size = len(df) * 0.8 # ฐานสำหรับ Proportional Sampling

print(f"Starting K-Fold Cross Validation (k={n_splits})")
print("=" * 100)

for fold_idx, (train_idx, test_idx) in enumerate(skf.split(df, df['Status']), start=1):
    train_fold = df.iloc[train_idx]
    test_fold  = df.iloc[test_idx]
    
    # 1. Proportional Sampling: ปรับลด 2000 ตามขนาด Fold
    adj_reserved = int(2000 * (len(train_fold) / base_train_size))
    train_cls_f, train_reg_f, reserved_low_f = make_v2_training_sets(train_fold, n_reserved=adj_reserved)

    X_test_fold     = test_fold.drop(columns=['Percent_Bleaching','Status']).reset_index(drop=True)
    y_test_fold     = test_fold['Percent_Bleaching'].reset_index(drop=True)
    y_test_cls_fold = (y_test_fold <= 1).values # True=Low (0), False=High (1)

    # --- Training Stage ---
    # Pipeline A: Classification
    X_train_cls = train_cls_f.drop(columns=['Percent_Bleaching','Status'])
    y_train_cls = train_cls_f['Status'].reset_index(drop=True)
    imp_A, sc_A, pc_A_, pcA_cols, X_tr_cls = fit_pipeline(X_train_cls)
    cls_fold = clone(rf_classifier)
    cls_fold.fit(X_tr_cls, y_train_cls)

    # Pipeline B: Regression
    X_train_reg = train_reg_f.drop(columns=['Percent_Bleaching','Status'])
    y_train_reg = train_reg_f['Percent_Bleaching'].reset_index(drop=True)
    imp_B, sc_B, pc_B_, pcB_cols, X_tr_reg = fit_pipeline(X_train_reg)

    fold_regs = {
        'HistGradientBoost': clone(hgb_model).fit(X_tr_reg, y_train_reg),
        'ExtraTrees':        clone(et_model).fit(X_tr_reg, y_train_reg),
        'RandomForest':      clone(rf_model).fit(X_tr_reg, y_train_reg),
    }
    voting_fold = VotingRegressor([('hgb', fold_regs['HistGradientBoost']), ('et', fold_regs['ExtraTrees']), ('rf', fold_regs['RandomForest'])]).fit(X_tr_reg, y_train_reg)

    # --- Prediction ---
    pipeA = (imp_A, sc_A, pc_A_, pcA_cols)
    pipeB = (imp_B, sc_B, pc_B_, pcB_cols)
    results, cls_pred, _ = predict_combined(X_test_fold, cls_fold, fold_regs, voting_fold, pipeA, pipeB)

    # --- Fold Reporting ---
    tn, fp, fn, tp = confusion_matrix(y_test_cls_fold, cls_pred).ravel()
    cls_acc = (cls_pred == y_test_cls_fold).mean()
    
    print(f"\n>>> FOLD {fold_idx} REPORT")
    print(f"Classification -> Accuracy: {cls_acc:.4f} | CM: [TN={tn}, FP={fp}, FN={fn}, TP={tp}]")
    print(f"{'Method':<22} | {'Overall RMSE':>12} | {'Pred 0 RMSE':>12} | {'NZ RMSE':>10} | {'NZ R2':>8}")
    print("-" * 85)

    for name, y_pred in results.items():
        # คำนวณ Metrics แยกกลุ่ม
        mask_0 = (cls_pred == True)
        mask_nz = (cls_pred == False)
        
        rmse_all = np.sqrt(mean_squared_error(y_test_fold, y_pred))
        rmse_0 = np.sqrt(mean_squared_error(y_test_fold[mask_0], y_pred[mask_0])) if mask_0.any() else 0
        
        rmse_nz, r2_nz = 0, 0
        if mask_nz.any():
            rmse_nz = np.sqrt(mean_squared_error(y_test_fold[mask_nz], y_pred[mask_nz]))
            if len(y_test_fold[mask_nz]) > 1:
                r2_nz = r2_score(y_test_fold[mask_nz], y_pred[mask_nz])

        print(f"{name:<22} | {rmse_all:>12.4f} | {rmse_0:>12.4f} | {rmse_nz:>10.4f} | {r2_nz:>8.4f}")
        
        # เก็บค่าไว้สรุปตอนท้าย
        final_scores[name]['r2'].append(r2_score(y_test_fold, y_pred))
        final_scores[name]['rmse'].append(rmse_all)
        final_scores[name]['rmse_pred_0'].append(rmse_0)
        final_scores[name]['rmse_pred_nonzero'].append(rmse_nz)
        final_scores[name]['r2_pred_nonzero'].append(r2_nz)

    cls_acc_history.append(cls_acc)
    conf_matrix_history.append([tn, fp, fn, tp])

# ===== FINAL SUMMARY =====
print("\n" + "=" * 100)
print("FINAL K-FOLD AVERAGE SUMMARY")
print("=" * 100)
avg_cm = np.mean(conf_matrix_history, axis=0)
print(f"Mean Cls Accuracy: {np.mean(cls_acc_history):.4f}")
print(f"Mean Confusion Matrix: TN={avg_cm[0]:.1f}, FP={avg_cm[1]:.1f}, FN={avg_cm[2]:.1f}, TP={avg_cm[3]:.1f}")
print("-" * 100)
for name in method_names:
    s = final_scores[name]
    print(f"{name:<22} | Avg RMSE: {np.mean(s['rmse']):.4f} | Avg NZ R2: {np.mean(s['r2_pred_nonzero']):.4f}")

Starting K-Fold Cross Validation (k=5)

>>> FOLD 1 REPORT
Classification -> Accuracy: 0.9508 | CM: [TN=1841, FP=127, FN=89, TP=2329]
Method                 | Overall RMSE |  Pred 0 RMSE |    NZ RMSE |    NZ R2
-------------------------------------------------------------------------------------
HistGradientBoost      |      13.3426 |       6.4113 |    18.7687 |   0.4979
ExtraTrees             |      12.0567 |       6.4113 |    16.6745 |   0.6037
RandomForest           |      12.2144 |       6.4113 |    16.9333 |   0.5913
Voting (Ensemble)      |      12.1877 |       6.4113 |    16.8895 |   0.5934
Sens-Weighted (per-row) |      12.4329 |       6.4113 |    17.2908 |   0.5738

>>> FOLD 2 REPORT
Classification -> Accuracy: 0.9448 | CM: [TN=1848, FP=120, FN=122, TP=2296]
Method                 | Overall RMSE |  Pred 0 RMSE |    NZ RMSE |    NZ R2
-------------------------------------------------------------------------------------
HistGradientBoost      |      13.8271 |       7.2327 |    19